# SBM Salutary Duplicity Explorer

Interactive notebook for the two-block SBM version of the Salutary Duplicity model.

**Sections**
1. Setup
2. Network + priors + payoff curves
3. Glauber dynamics animation + site occupancy trace
4. Live `p`-sweep: payoff vs lying level (sparse vs abundant)

Works on Colab (uploads `src/` next to the notebook, or clones the repo) and locally via the project venv.

## Variables

**Network (Section 1)**
- `N1`, `N2` — sizes of the two SBM blocks (block 0 = "expert", block 1 = "naive").
- `p_in` — within-block edge probability. Higher = denser communities.
- `p_out` — between-block edge probability. Higher = more cross-talk between experts and naive.
- `seed` — RNG seed for graph + priors + simulation.

**Site qualities and forage curves**
- `q_A`, `q_B` — intrinsic productivity of sites A and B. Used as `C = [q_A, q_B]` in the payoff `mu_k = C_k / (1 + exp(gamma * (delta*N_k - F_k)))`.
- `F_A`, `F_B` — carrying capacities. Per-agent payoff stays near `q_k` while `delta*N_k < F_k`, drops off sigmoidally past it.
- `gamma` — steepness of the congestion roll-off. Larger = sharper crash when capacity is exceeded.
- `delta` — per-agent forage demand. Higher = each agent "costs more" capacity.

**Priors**
- `sigma_expert` — std of the expert block's prior. Experts are centered at `log(q_A / q_B)` (i.e. correctly biased toward the better site).
- `sigma_naive` — std of the naive block's prior. Naive agents are centered at 0 (no informed bias).

**Glauber dynamics (Section 2)**
- `beta` — inverse temperature. Higher = more deterministic spin flips, agents follow the effective field more closely.
- `J_in` — coupling strength on within-block edges (expert↔expert and naive↔naive). Larger = stronger conformity inside each community.
- `J_out` — coupling strength on between-block edges (expert↔naive). Larger = stronger cross-community influence; set lower than `J_in` to encode community insularity.
- `alpha` — per-agent environmental-learning weight in [0,1]. `0` = pure social learning, `1` = pure individual learning of the quality gap.
- `p_expert` — honesty probability for the expert block. `1` = experts always honest, `0` = experts always lie.
- `p_naive` — honesty probability for the naive block. Same convention.
- `T` — number of phase-A sweeps to animate (one sweep = N agent updates).

**Live `p`-sweep (Section 3)**
- `N` — population size (a fresh Erdős–Rényi graph is generated each recompute).
- `F_ratio_sparse`, `F_ratio_abundant` — sets `F_total = ratio * N`, split evenly across the two sites. `ratio < 1` is scarce forage, `ratio > 1` is plentiful.
- `strategy_alpha` — replicator learning rate for the Phase C macro update.
- `cost` — `individual_learning_cost`: payoff penalty per unit of `alpha`.
- `n_epochs` — full Phase A→B→C epochs per simulation.
- `n_seeds` — independent seeds per p-value (curves show ±1 std).
- `n_p` — number of p-values swept across [0, 1].

Sliders in Section 3 have `continuous_update=False`, so changes fire only on mouse release.


## Section 0 — Setup

In [ ]:
import sys, os, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'networkx', 'matplotlib', 'ipywidgets', 'numpy'], check=True)
    # If the repo isn't already cloned, clone it so we can import salutary_duplicity.
    if not os.path.isdir('SalutaryDuplicity'):
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/WillHWThompson/SalutaryDuplicity.git'],
                       check=False)
    if os.path.isdir('SalutaryDuplicity/src'):
        sys.path.insert(0, 'SalutaryDuplicity/src')
else:
    # Local: assume notebooks/ sits next to src/
    sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.animation as manim
from matplotlib.colors import TwoSlopeNorm
import ipywidgets as widgets
from IPython.display import HTML, display

from salutary_duplicity.model import SalutaryModel
%matplotlib inline
print('Imports OK. salutary_duplicity from:', SalutaryModel.__module__)

In [ ]:
DEFAULTS = {
    'N1': 40, 'N2': 40,
    'p_in': 0.15, 'p_out': 0.02,
    'q_A': 1.5, 'q_B': 1.0,
    'sigma_expert': 0.3, 'sigma_naive': 0.5,
    'gamma': 2.0, 'delta': 1.0,
    'F_A': 40.0, 'F_B': 40.0,
    'beta': 1.0, 'J_strength': 0.5, 'alpha': 0.0,
    'T': 60, 'seed': 0,
    'n_p': 15, 'n_seeds': 3, 'n_epochs': 30,
    'n_sweeps_anim': 1, 'n_sweeps_sweep': 20,
}

def build_sbm(N1, N2, p_in, p_out, seed):
    sizes = [N1, N2]
    P = [[p_in, p_out], [p_out, p_in]]
    G = nx.stochastic_block_model(sizes, P, seed=seed)
    G = nx.convert_node_labels_to_integers(G)
    blocks = np.concatenate([np.zeros(N1, dtype=int), np.ones(N2, dtype=int)])
    return G, blocks

def build_priors(blocks, q_A, q_B, sigma_expert, sigma_naive, seed):
    rng = np.random.default_rng(seed)
    center = np.log(q_A / q_B)
    h = np.empty(len(blocks))
    expert_mask = blocks == 0
    naive_mask = blocks == 1
    h[expert_mask] = rng.normal(center, sigma_expert, size=expert_mask.sum())
    h[naive_mask] = rng.normal(0.0, sigma_naive, size=naive_mask.sum())
    return h

def site_payoff(N_k, F_k, C_k, gamma, delta):
    return C_k / (1.0 + np.exp(gamma * (delta * N_k - F_k)))

## Section 1 — Network, priors, payoff curves

In [ ]:
s1 = {
    'N1': widgets.IntSlider(value=DEFAULTS['N1'], min=10, max=100, step=5, description='N1 (expert)'),
    'N2': widgets.IntSlider(value=DEFAULTS['N2'], min=10, max=100, step=5, description='N2 (naive)'),
    'p_in': widgets.FloatSlider(value=DEFAULTS['p_in'], min=0.0, max=0.6, step=0.01, description='p_in'),
    'p_out': widgets.FloatSlider(value=DEFAULTS['p_out'], min=0.0, max=0.3, step=0.005, description='p_out'),
    'q_A': widgets.FloatSlider(value=DEFAULTS['q_A'], min=0.5, max=3.0, step=0.05, description='q_A'),
    'q_B': widgets.FloatSlider(value=DEFAULTS['q_B'], min=0.5, max=3.0, step=0.05, description='q_B'),
    'sigma_expert': widgets.FloatSlider(value=DEFAULTS['sigma_expert'], min=0.0, max=1.0, step=0.05, description='σ expert'),
    'sigma_naive': widgets.FloatSlider(value=DEFAULTS['sigma_naive'], min=0.0, max=1.5, step=0.05, description='σ naive'),
    'gamma': widgets.FloatSlider(value=DEFAULTS['gamma'], min=0.1, max=8.0, step=0.1, description='γ'),
    'delta': widgets.FloatSlider(value=DEFAULTS['delta'], min=0.1, max=3.0, step=0.1, description='δ'),
    'F_A': widgets.FloatSlider(value=DEFAULTS['F_A'], min=5.0, max=200.0, step=1.0, description='F_A'),
    'F_B': widgets.FloatSlider(value=DEFAULTS['F_B'], min=5.0, max=200.0, step=1.0, description='F_B'),
    'seed': widgets.IntSlider(value=DEFAULTS['seed'], min=0, max=20, step=1, description='seed'),
}

def draw_section1(N1, N2, p_in, p_out, q_A, q_B, sigma_expert, sigma_naive,
                  gamma, delta, F_A, F_B, seed):
    G, blocks = build_sbm(N1, N2, p_in, p_out, seed)
    h = build_priors(blocks, q_A, q_B, sigma_expert, sigma_naive, seed)
    pos = nx.spring_layout(G, seed=seed)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    vmax = max(1e-6, np.abs(h).max())
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
    for blk, marker in [(0, 'o'), (1, 's')]:
        nodes = [n for n, b in enumerate(blocks) if b == blk]
        nx.draw_networkx_nodes(G, pos, nodelist=nodes, node_color=h[nodes],
                               cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                               node_shape=marker, node_size=80,
                               edgecolors='black', linewidths=0.4, ax=axes[0])
    nx.draw_networkx_edges(G, pos, alpha=0.15, ax=axes[0])
    axes[0].set_title(f'SBM (N={N1+N2}, ●=expert ■=naive)')
    axes[0].axis('off')

    axes[1].hist(h[blocks==0], bins=20, alpha=0.6, label='expert', color='C0')
    axes[1].hist(h[blocks==1], bins=20, alpha=0.6, label='naive', color='C1')
    axes[1].axvline(np.log(q_A/q_B), ls='--', color='k', lw=1.0,
                    label=f'log(q_A/q_B)={np.log(q_A/q_B):.2f}')
    axes[1].axvline(0.0, ls=':', color='gray', lw=1.0)
    axes[1].set_xlabel('h_i'); axes[1].set_ylabel('count')
    axes[1].set_title('Priors per block'); axes[1].legend()

    N_total = N1 + N2
    Nk_grid = np.linspace(0, N_total, 200)
    muA = site_payoff(Nk_grid, F_A, q_A, gamma, delta)
    muB = site_payoff(Nk_grid, F_B, q_B, gamma, delta)
    axes[2].plot(Nk_grid, muA, color='C0', label=f'site A (q={q_A:.2f})')
    axes[2].plot(Nk_grid, muB, color='C3', label=f'site B (q={q_B:.2f})')
    axes[2].axvline(F_A, color='C0', ls='--', alpha=0.6)
    axes[2].axvline(F_B, color='C3', ls='--', alpha=0.6)
    axes[2].set_xlabel('N_k (agents at site)'); axes[2].set_ylabel('per-agent payoff μ_k')
    axes[2].set_title('Sigmoid forage curves'); axes[2].legend()

    plt.tight_layout(); plt.show()

s1_ui = widgets.VBox([
    widgets.HBox([s1['N1'], s1['N2'], s1['p_in'], s1['p_out']]),
    widgets.HBox([s1['q_A'], s1['q_B'], s1['sigma_expert'], s1['sigma_naive']]),
    widgets.HBox([s1['gamma'], s1['delta'], s1['F_A'], s1['F_B'], s1['seed']]),
])
s1_out = widgets.interactive_output(draw_section1, s1)
display(s1_ui, s1_out)

## Section 2 — Glauber dynamics animation

Press the play button on the scrub bar. Phase C (strategy update) is disabled here so you can watch the spin dynamics alone. The right panel shows site occupancy vs the carrying capacities.

In [ ]:
s2 = {
    'beta': widgets.FloatSlider(value=DEFAULTS['beta'], min=0.1, max=4.0, step=0.05, description='beta'),
    'J_in': widgets.FloatSlider(value=DEFAULTS['J_strength'], min=0.0, max=2.0, step=0.05, description='J within'),
    'J_out': widgets.FloatSlider(value=DEFAULTS['J_strength'], min=0.0, max=2.0, step=0.05, description='J between'),
    'alpha': widgets.FloatSlider(value=DEFAULTS['alpha'], min=0.0, max=1.0, step=0.05, description='alpha'),
    'p_expert': widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05, description='p expert'),
    'p_naive': widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05, description='p naive'),
    'T': widgets.IntSlider(value=DEFAULTS['T'], min=10, max=80, step=5, description='T sweeps'),
}
run_btn = widgets.Button(description='Run animation', button_style='primary')
anim_out = widgets.Output()

def _build_J_matrix(blocks, J_in, J_out):
    N = len(blocks)
    same = (blocks[:, None] == blocks[None, :])
    return np.where(same, J_in, J_out).astype(float)

def _build_p_vector(blocks, p_expert, p_naive):
    out = np.where(blocks == 0, p_expert, p_naive).astype(float)
    return out

def _run_anim(_):
    anim_out.clear_output(wait=True)
    with anim_out:
        N1, N2 = s1['N1'].value, s1['N2'].value
        G, blocks = build_sbm(N1, N2, s1['p_in'].value, s1['p_out'].value, s1['seed'].value)
        h = build_priors(blocks, s1['q_A'].value, s1['q_B'].value,
                         s1['sigma_expert'].value, s1['sigma_naive'].value, s1['seed'].value)
        N = N1 + N2
        J_mat = _build_J_matrix(blocks, s2['J_in'].value, s2['J_out'].value)
        p_vec = _build_p_vector(blocks, s2['p_expert'].value, s2['p_naive'].value)
        params = {
            'C': [s1['q_A'].value, s1['q_B'].value],
            'FA': [s1['F_A'].value, s1['F_B'].value],
            'gamma': s1['gamma'].value, 'delta': s1['delta'].value,
            'beta': s2['beta'].value, 'J_strength': J_mat,
            'alpha': s2['alpha'].value, 'n_sweeps': 1,
            'update_mode': 'none',
        }
        model = SalutaryModel(G, h, strategies=(h > 0), p=p_vec,
                              params=params, seed=s1['seed'].value)
        T = s2['T'].value
        frames = [model.spins.copy()]
        for _t in range(T):
            model.run_phase_a()
            frames.append(model.spins.copy())
        frames = np.array(frames)

        pos = nx.spring_layout(G, seed=s1['seed'].value)
        fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5.5))
        coords = np.array([pos[n] for n in range(N)])
        colors0 = np.where(frames[0] > 0, '#1f77b4', '#d62728')
        markers = np.where(blocks == 0, 'o', 's')
        # scatter twice so block shapes are preserved
        idx0 = np.where(blocks == 0)[0]; idx1 = np.where(blocks == 1)[0]
        scat0 = axL.scatter(coords[idx0, 0], coords[idx0, 1], c=colors0[idx0],
                            marker='o', s=70, edgecolor='black', linewidth=0.4, zorder=3)
        scat1 = axL.scatter(coords[idx1, 0], coords[idx1, 1], c=colors0[idx1],
                            marker='s', s=70, edgecolor='black', linewidth=0.4, zorder=3)
        for u, v in G.edges():
            axL.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]],
                     color='lightgray', lw=0.3, zorder=1)
        axL.set_xticks([]); axL.set_yticks([])
        title = axL.set_title(f't=0 / {T} (blue=A, red=B; o=expert, s=naive)')

        N_A_t = (frames > 0).sum(axis=1)
        N_B_t = (frames < 0).sum(axis=1)
        ts = np.arange(len(frames))
        axR.plot(ts, N_A_t, color='#1f77b4', label='N_A')
        axR.plot(ts, N_B_t, color='#d62728', label='N_B')
        axR.axhline(s1['F_A'].value, ls='--', color='#1f77b4', alpha=0.6, label=f'F_A={s1["F_A"].value:.0f}')
        axR.axhline(s1['F_B'].value, ls='--', color='#d62728', alpha=0.6, label=f'F_B={s1["F_B"].value:.0f}')
        vline = axR.axvline(0, color='k', lw=0.7)
        axR.set_xlabel('phase-A sweep'); axR.set_ylabel('agents at site')
        axR.set_title('Site occupancy vs carrying capacity'); axR.legend(loc='best', fontsize=8)

        def update(t):
            c = np.where(frames[t] > 0, '#1f77b4', '#d62728')
            scat0.set_color(c[idx0])
            scat1.set_color(c[idx1])
            title.set_text(f't={t} / {T}')
            vline.set_xdata([t, t])
            return scat0, scat1, title, vline

        ani = manim.FuncAnimation(fig, update, frames=len(frames), interval=120, blit=False)
        plt.close(fig)
        display(HTML(ani.to_jshtml()))

run_btn.on_click(_run_anim)
display(widgets.VBox([
    widgets.HBox([s2['beta'], s2['alpha'], s2['T']]),
    widgets.HBox([s2['J_in'], s2['J_out']]),
    widgets.HBox([s2['p_expert'], s2['p_naive'], run_btn]),
    anim_out,
]))


## Section 3 — Live `p`-sweep: payoff vs lying level

Sweeps over `p ∈ [0, 1]` on every recompute. The other parameters are sliders. Two curves are drawn: sparse forage (`F_total = ratio_sparse × N`) and abundant forage (`F_total = ratio_abundant × N`). The salutary-duplicity peak shows up as an interior maximum on the sparse curve.

In [ ]:
def _f(value, mn, mx, step, desc):
    return widgets.FloatSlider(value=value, min=mn, max=mx, step=step,
                               description=desc, continuous_update=False)
def _i(value, mn, mx, step, desc):
    return widgets.IntSlider(value=value, min=mn, max=mx, step=step,
                             description=desc, continuous_update=False)
s3 = {
    'N': _i(80, 40, 160, 10, 'N'),
    'beta': _f(1.0, 0.1, 4.0, 0.05, 'beta'),
    'gamma': _f(2.0, 0.5, 6.0, 0.1, 'gamma'),
    'delta': _f(1.0, 0.5, 2.0, 0.05, 'delta'),
    'q_A': _f(1.5, 0.5, 3.0, 0.05, 'q_A'),
    'q_B': _f(1.0, 0.5, 3.0, 0.05, 'q_B'),
    'F_ratio_sparse': _f(0.5, 0.2, 1.5, 0.05, 'F ratio sparse'),
    'F_ratio_abundant': _f(2.0, 1.0, 3.0, 0.1, 'F ratio abundant'),
    'strategy_alpha': _f(0.05, 0.0, 0.5, 0.01, 'strat alpha'),
    'cost': _f(0.0, 0.0, 1.0, 0.05, 'ind. cost'),
    'alpha': _f(0.0, 0.0, 1.0, 0.05, 'alpha (env wt)'),
    'n_epochs': _i(30, 5, 80, 5, 'epochs'),
    'n_seeds': _i(3, 1, 8, 1, 'seeds'),
    'n_p': _i(15, 5, 25, 1, '# p-values'),
}

def _one_payoff(N, p_val, F_total, q_A, q_B, beta, gamma, delta,
                strategy_alpha, cost, alpha, n_epochs, seed):
    G = nx.erdos_renyi_graph(N, 0.1, seed=seed)
    rng = np.random.default_rng(seed)
    h = rng.normal(np.log(q_A / q_B) * 0.5, 0.4, size=N)
    params = {
        'C': [q_A, q_B], 'FA': [F_total/2, F_total/2],
        'gamma': gamma, 'delta': delta, 'beta': beta,
        'alpha': alpha, 'strategy_alpha': strategy_alpha,
        'individual_learning_cost': cost,
        'n_sweeps': DEFAULTS['n_sweeps_sweep'],
        'update_mode': 'macro',
    }
    model = SalutaryModel(G, h, strategies=(h > 0),
                          p=np.full(N, p_val), params=params, seed=seed)
    last = None
    for _ in range(n_epochs):
        last = model.run_epoch()
    return last['mean_payoff']

def run_sweep(N, beta, gamma, delta, q_A, q_B,
              F_ratio_sparse, F_ratio_abundant,
              strategy_alpha, cost, alpha, n_epochs, n_seeds, n_p):
    p_grid = np.linspace(0.0, 1.0, n_p)
    regimes = [('sparse', F_ratio_sparse, '#3a6d91'),
               ('abundant', F_ratio_abundant, '#c98a3f')]
    fig, ax = plt.subplots(figsize=(8, 5))
    sparse_curve = None
    for label, ratio, color in regimes:
        F_total = ratio * N
        means = np.zeros(n_p); stds = np.zeros(n_p)
        for i, p_val in enumerate(p_grid):
            vals = [
                _one_payoff(N, p_val, F_total, q_A, q_B, beta, gamma, delta,
                            strategy_alpha, cost, alpha, n_epochs, seed)
                for seed in range(n_seeds)
            ]
            means[i] = np.mean(vals); stds[i] = np.std(vals)
        lying = 1.0 - p_grid
        order = np.argsort(lying)
        x = lying[order]; m = means[order]; s = stds[order]
        ax.plot(x, m, color=color, lw=2.4, label=f'{label} (F/N={ratio:.2f})')
        ax.fill_between(x, m-s, m+s, color=color, alpha=0.18)
        if label == 'sparse':
            sparse_curve = (x, m)
    if sparse_curve is not None:
        x, m = sparse_curve
        k = int(np.argmax(m))
        if 0 < k < len(m) - 1:
            ax.axvline(x[k], color='#3a6d91', ls=':', alpha=0.7,
                       label=f'sparse peak at lying={x[k]:.2f}')
    ax.set_xlabel('initial lying level (1 - p)')
    ax.set_ylabel('mean per-agent payoff (final epoch)')
    ax.set_title('Payoff vs lying level - live recompute')
    ax.legend(loc='best')
    plt.tight_layout(); plt.show()

s3_ui = widgets.VBox([
    widgets.HBox([s3['N'], s3['n_p'], s3['n_seeds'], s3['n_epochs']]),
    widgets.HBox([s3['beta'], s3['gamma'], s3['delta']]),
    widgets.HBox([s3['q_A'], s3['q_B'], s3['F_ratio_sparse'], s3['F_ratio_abundant']]),
    widgets.HBox([s3['strategy_alpha'], s3['cost'], s3['alpha']]),
])
s3_out = widgets.interactive_output(run_sweep, s3)
display(s3_ui, s3_out)
